# AgentWonder Phase 2 — Template Pattern Flows

This notebook exercises all five workflow template patterns:

1. **Sequential with Approval** — `break_resolution_v1`
2. **Single Agent with Tools** — `customer_support_agent_v1`
3. **Router → Specialists** — `ticket_router_v1`
4. **Parallel Fanout → Aggregate** — `market_data_fanout_v1`
5. **Evaluator Loop** — `content_review_loop_v1`

Plus: enhanced tracing, CLI validation, and approval timeout checks.

In [ ]:
import os, json
#os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from pathlib import Path
from agentwonder.compiler.loader import load_yaml, load_all_yaml
from agentwonder.compiler.validators import validate_workflow, cross_validate_workflow
from agentwonder.compiler.resolver import resolve_workflow
from agentwonder.compiler.builder import build_plan
from agentwonder.registry import TemplateRegistry, ToolRegistry, PromptRegistry, PolicyRegistry
from agentwonder.runtime.executor import WorkflowExecutor
from agentwonder.schemas.run import RunRequest

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

config = Path("config")
tr = TemplateRegistry(); tr.load_from_directory(config / "templates")
tl = ToolRegistry(); tl.load_from_directory(config / "tools")
pr = PromptRegistry(); pr.load_from_directory(config / "prompts")
po = PolicyRegistry(); po.load_from_directory(config / "policies")

tools_d = {t.id: t for t in tl.list_all()}
templates_d = {t.id: t for t in tr.list_all()}
prompts_d = {t.id: t for t in pr.list_all()}
policies_d = {t.id: t for t in po.list_all()}

print(f"Templates: {sorted(templates_d.keys())}")
print(f"Tools: {len(tools_d)}, Prompts: {len(prompts_d)}, Policies: {len(policies_d)}")

def pipeline(wf_file, inputs):
    raw = load_yaml(config / "workflows" / wf_file)
    wf = validate_workflow(raw)
    errors = cross_validate_workflow(wf, tools_d, templates_d, policies_d, prompts_d)
    assert not errors, f"Errors: {errors}"
    resolved = resolve_workflow(wf, tools_d, templates_d, prompts_d, policies_d)
    return build_plan(resolved), RunRequest(workflow_id=wf.id, inputs=inputs)

executor = WorkflowExecutor()

GOOGLE_API_KEY not set — LLM calls will use stub responses


Templates: ['evaluator_loop', 'parallel_fanout_aggregate', 'router_specialists', 'sequential_with_approval', 'single_agent_with_tools']
Tools: 13, Prompts: 2, Policies: 2


## 1. Sequential with Approval — `break_resolution_v1`

In [ ]:
plan, req = pipeline("break_resolution_v1.yaml", {"break_id": "BRK-42", "source_system": "ops"})
status = await executor.execute(req, plan)
print(f"[sequential_with_approval] {status.state.value} | steps: {list(status.outputs.keys())}")
assert status.state.value == "completed"

[sequential_with_approval] completed | steps: ['enrich_break', 'enrich_trade_context', 'propose_resolution', 'checker_approval', 'update_downstream']


## 2. Single Agent with Tools — `customer_support_agent_v1`

In [ ]:
plan, req = pipeline("customer_support_agent_v1.yaml", {"query": "I need help with my account"})
status = await executor.execute(req, plan)
print(f"[single_agent_with_tools] {status.state.value} | steps: {list(status.outputs.keys())}")
assert status.state.value == "completed"

[single_agent_with_tools] completed | steps: ['handle_query', 'log_result']


## 3. Router → Specialists — `ticket_router_v1`

In [ ]:
plan, req = pipeline("ticket_router_v1.yaml", {"ticket_id": "T-200"})
status = await executor.execute(req, plan)
print(f"[router_specialists] {status.state.value} | steps: {list(status.outputs.keys())}")
# Show router decision
router_out = status.outputs.get("route_ticket", {})
print(f"  Router selected: {router_out.get('selected_route')}")
print(f"  Aggregator result: {status.outputs.get('aggregate_response', {}).get('branch_count')} branches")
assert status.state.value == "completed"

[router_specialists] completed | steps: ['route_ticket', 'billing_specialist', 'technical_specialist', 'general_specialist', 'aggregate_response']
  Router selected: billing_specialist
  Aggregator result: 3 branches


## 4. Parallel Fanout → Aggregate — `market_data_fanout_v1`

In [ ]:
plan, req = pipeline("market_data_fanout_v1.yaml", {"source": "live"})
print(f"  Parallel groups: {plan.parallel_groups}")
status = await executor.execute(req, plan)
print(f"[parallel_fanout_aggregate] {status.state.value} | steps: {list(status.outputs.keys())}")
agg = status.outputs.get("combine_snapshot", {})
print(f"  Aggregated from: {agg.get('aggregated_from')}")
assert status.state.value == "completed"

  Parallel groups: [['fanout'], ['combine_snapshot']]
[parallel_fanout_aggregate] completed | steps: ['fanout', 'combine_snapshot']
  Aggregated from: ['fanout']


## 5. Evaluator Loop — `content_review_loop_v1`

In [ ]:
plan, req = pipeline("content_review_loop_v1.yaml", {"topic": "AI governance"})
status = await executor.execute_with_eval_loop(
    req, plan,
    generator_step_id="generate_draft",
    evaluator_step_id="evaluate_quality",
    max_iterations=3,
)
print(f"[evaluator_loop] {status.state.value} | steps: {list(status.outputs.keys())}")
session = executor.session_store.get_session(status.run_id)
loop_events = [e for e in session["data"]["trace_events"] if "eval_loop" in e["event_type"]]
for evt in loop_events:
    print(f"  {evt['event_type']}: {evt['data']}")
assert status.state.value == "completed"

[evaluator_loop] completed | steps: ['generate_draft', 'evaluate_quality', 'publish']
  eval_loop_iteration: {'iteration': 1, 'max_iterations': 3}
  eval_loop_result: {'iteration': 1, 'passed': True, 'score': 0.95}


## 6. Enhanced Tracing — Per-Step Timings and Model Info

In [ ]:
# Reuse the sequential workflow for detailed trace inspection
plan, req = pipeline("break_resolution_v1.yaml", {"break_id": "BRK-99", "source_system": "test"})
status = await executor.execute(req, plan)
session = executor.session_store.get_session(status.run_id)
events = session["data"]["trace_events"]

print("Trace event summary:\n")
for evt in events:
    step = evt.get("step_id") or "(run-level)"
    etype = evt["event_type"]
    data = evt.get("data", {})
    extra = ""
    if "duration_ms" in data:
        extra = f"  duration={data['duration_ms']}ms"
    elif "model_default" in data:
        extra = f"  model={data['model_default']}"
    elif "tool_id" in data:
        extra = f"  tool={data['tool_id']} v{data.get('tool_version', '?')}"
    print(f"  [{etype:>22}]  step={step:<25}{extra}")

# Verify timings exist
step_ends = [e for e in events if e["event_type"] == "step_end"]
assert all("duration_ms" in e["data"] for e in step_ends), "Missing duration_ms in step_end events"
print(f"\nAll {len(step_ends)} step_end events have duration_ms")

Trace event summary:

  [             run_start]  step=(run-level)                model=gemini-2.5-flash
  [            step_start]  step=enrich_break             
  [          tool_invoked]  step=enrich_break               tool=fetch_break_details v1.0.0
  [              step_end]  step=enrich_break               duration=0.1ms
  [            step_start]  step=enrich_trade_context     
  [          tool_invoked]  step=enrich_trade_context       tool=fetch_trade_context v1.0.0
  [              step_end]  step=enrich_trade_context       duration=0.02ms
  [            step_start]  step=propose_resolution       
  [           llm_invoked]  step=propose_resolution       
  [              step_end]  step=propose_resolution         duration=0.04ms
  [            step_start]  step=checker_approval         
  [    approval_requested]  step=checker_approval         
  [      approval_decided]  step=checker_approval         
  [              step_end]  step=checker_approval           duration=0.

## 7. Summary — All Assertions

In [ ]:
# Validate ALL workflows pass cross-validation
all_raw = load_all_yaml(config / "workflows")
print(f"Validating {len(all_raw)} workflows...\n")
for raw in all_raw:
    wf = validate_workflow(raw)
    errors = cross_validate_workflow(wf, tools_d, templates_d, policies_d, prompts_d)
    status_str = "PASS" if not errors else f"FAIL ({len(errors)} errors)"
    print(f"  {wf.id:40s} [{status_str}]")
    assert not errors, f"{wf.id}: {errors}"

print("\nAll workflows validated successfully!")
print("Phase 2 notebook complete — all 5 template patterns executed.")

Validating 5 workflows...

  break_resolution_v1                      [PASS]
  content_review_loop_v1                   [PASS]
  customer_support_agent_v1                [PASS]
  market_data_fanout_v1                    [PASS]
  ticket_router_v1                         [PASS]

All workflows validated successfully!
Phase 2 notebook complete — all 5 template patterns executed.
